# Module 7: Knowledge Distillation

> **Course: Training MiniMind LLM from Scratch**  |  *Google Colab NLP Course*

## 🎯 Learning Objectives
- Understand **knowledge distillation** and when to use it
- Implement the **combined distillation loss** from scratch
- Train a student MiniMind model supervised by a teacher
- Visualise the effect of **temperature scaling** on probability distributions

---

## 🧠 What is Knowledge Distillation?

**Goal:** Transfer knowledge from a large *teacher* model into a smaller *student* model.

### Two paradigms
| Approach | Teacher access | What is transferred |
|----------|---------------|---------------------|
| **Black-box** | Output text only | Final predictions / answers |
| **White-box** | Full logits | Soft probability distributions ← *we use this* |

### Why soft labels are better

Hard labels are one-hot: `[0, 0, 1, 0, 0]` — the model either gets it right or wrong.

Teacher soft labels carry **richer information**:  
`[0.01, 0.05, 0.82, 0.09, 0.03]` — "82% confident in token 2, but token 3 is plausible too."

This *dark knowledge* encodes **semantic relationships** the student can learn from.

### The Distillation Loss

$$\mathcal{L} = \alpha \mathcal{L}_{CE} + (1-\alpha) \cdot T^2 \cdot \text{KL}\!\left(p_T^{(T)} \,\|\, p_S^{(T)}\right)$$

- $\mathcal{L}_{CE}$: standard cross-entropy against ground-truth labels (hard loss)
- $\text{KL}(\cdot\|\cdot)$: KL divergence between teacher and student **at temperature $T$**
- $T^2$ compensates for the gradient magnitude reduction caused by temperature scaling
- $\alpha \in [0,1]$ balances the two objectives


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd): return subprocess.run(cmd, shell=True, check=False)

run("pip install -q torch transformers==4.46.3 modelscope tqdm matplotlib")

if not os.path.exists('/content/minimind-colab'):
    run("git clone https://github.com/Boyu-Zhang-UOI/minimind-colab /content/minimind-colab")

sys.path.insert(0, '/content/minimind-colab')

run("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU'")

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 🎓 The Distillation Approach

In our setup:
- **Teacher** = MiniMind SFT model (already trained in Module 5)
- **Student** = freshly initialised MiniMind (same architecture, trained from scratch)

Both models share the **same architecture** here — in practice the student would be smaller.  
The benefit even with equal sizes is that the teacher's *soft targets* act as a better training signal than one-hot labels.

In [ ]:
import argparse, os, torch

args = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='distill',
    teacher_weight='full_sft',
    data_path='/content/minimind-colab/dataset/sft_t2t_mini.jsonl',
    hidden_size=768, num_hidden_layers=8, use_moe=0,
    epochs=1, batch_size=8, learning_rate=5e-5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16', num_workers=2,
    accumulation_steps=1, grad_clip=1.0,
    log_interval=50, save_interval=500,
    max_seq_len=512,
    temperature=2.0,   # T  — controls softness of distributions
    alpha=0.5,         # α  — balance between hard and soft loss
)
os.makedirs(args.save_dir, exist_ok=True)
print("Distillation Configuration:")
for k, v in vars(args).items():
    print(f"  {k:<22} = {v}")


In [ ]:
import sys
sys.path.insert(0, '/content/minimind-colab')

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import SFTDataset
from trainer.trainer_utils import setup_seed, get_model_params, SkipBatchSampler, get_lr, init_model
from contextlib import nullcontext
from torch import optim
from torch.utils.data import DataLoader

setup_seed(42)
lm_config = MiniMindConfig(hidden_size=args.hidden_size,
                            num_hidden_layers=args.num_hidden_layers)

# ── Teacher (frozen) ───────────────────────────────────────────────────────
print("Loading teacher model …")
teacher, tokenizer = init_model(lm_config, args.teacher_weight,
                                 tokenizer_path='/content/minimind-colab/model',
                                 save_dir=args.save_dir, device=args.device)
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()
print("  ✅ Teacher loaded & frozen")

# ── Student (trainable) ────────────────────────────────────────────────────
print("Initialising student model …")
student = MiniMindForCausalLM(lm_config).to(args.device)
total_params = sum(p.numel() for p in student.parameters())
print(f"  ✅ Student initialised  ({total_params/1e6:.2f}M params, random weights)")

# ── Data / optimiser ───────────────────────────────────────────────────────
if not os.path.exists(args.data_path):
    !modelscope download --dataset gongjy/minimind_dataset sft_t2t_mini.jsonl         --local_dir /content/minimind-colab/dataset

train_ds      = SFTDataset(args.data_path, tokenizer, max_length=args.max_seq_len)
optimizer     = optim.AdamW(student.parameters(), lr=args.learning_rate, weight_decay=0.01)
dtype_t       = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
autocast_ctx  = (nullcontext() if 'cpu' in args.device
                 else torch.cuda.amp.autocast(dtype=dtype_t))
scaler        = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

print(f"Dataset: {len(train_ds):,} samples  |  Device: {args.device}")


In [ ]:
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, labels,
                      temperature=2.0, alpha=0.5):
    """
    Combined hard + soft distillation loss.

    Returns
    -------
    total_loss  : Tensor  — scalar loss for backward()
    hard_val    : float   — CE loss value (for logging)
    soft_val    : float   — KL loss value (for logging)
    """
    # Hard label loss — standard CE, ignore padding (-100)
    hard_loss = F.cross_entropy(
        student_logits.view(-1, student_logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )

    # Soft label loss — KL divergence at temperature T
    # Note: only over non-padding positions for efficiency
    valid = labels.view(-1) != -100
    if valid.sum() == 0:
        return hard_loss, hard_loss.item(), 0.0

    t_probs   = F.softmax(teacher_logits.view(-1, teacher_logits.size(-1))[valid] / temperature, dim=-1)
    s_logp    = F.log_softmax(student_logits.view(-1, student_logits.size(-1))[valid] / temperature, dim=-1)
    # T^2 compensates for reduced gradient magnitude from temperature scaling:
    # dividing logits by T shrinks gradients by 1/T^2, so we multiply back.
    soft_loss = F.kl_div(s_logp, t_probs, reduction='batchmean') * (temperature ** 2)

    total = alpha * hard_loss + (1 - alpha) * soft_loss
    return total, hard_loss.item(), soft_loss.item()

print("✅ distillation_loss() defined")
print(f"   Temperature T={args.temperature}  →  distributions are {'soft' if args.temperature > 1 else 'sharp'}")
print(f"   Alpha α={args.alpha}  →  {int(args.alpha*100)}% hard labels, {int((1-args.alpha)*100)}% soft (teacher) labels")


## 📊 Visualising Temperature Scaling

Higher temperature → flatter distribution → student learns more from teacher's **relative** probabilities.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

logits = np.array([2.0, 1.0, 0.5, -0.5, -1.0])
labels = ['tok A', 'tok B', 'tok C', 'tok D', 'tok E']
temps  = [0.5, 1.0, 2.0, 5.0]
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']

fig, axes = plt.subplots(1, len(temps), figsize=(16, 4), sharey=True)
for i, (T, col) in enumerate(zip(temps, colors)):
    probs = np.exp(logits / T) / np.exp(logits / T).sum()
    axes[i].bar(labels, probs, color=col, edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'T = {T}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Token')
    axes[i].set_ylim(0, 1)
    entropy = -np.sum(probs * np.log(probs + 1e-9))
    axes[i].set_xlabel(f'Token\nEntropy={entropy:.2f}')
    if i == 0:
        axes[i].set_ylabel('Probability')

plt.suptitle('Effect of Temperature on Softmax Distribution\n'
             '(Higher T = softer = more information for the student)',
             fontsize=12)
plt.tight_layout()
plt.show()

print("Observation: at T=0.5 the winner takes almost all probability.")
print("At T=5.0 all tokens get meaningful probability — student learns")
print("that token B, C are also plausible, which encodes semantic similarity.")


## 🏋️ Distillation Training Loop

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

loss_history, hard_history, soft_history, step_history = [], [], [], []

def distill_train(epochs=1):
    student.train()
    global_step = 0
    for epoch in range(epochs):
        indices       = torch.randperm(len(train_ds)).tolist()
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader        = DataLoader(train_ds, batch_sampler=batch_sampler,
                                    num_workers=args.num_workers, pin_memory=True)
        iters = len(loader)
        pbar  = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, (input_ids, labels) in enumerate(pbar, start=1):
            global_step += 1
            input_ids = input_ids.to(args.device)
            labels    = labels.to(args.device)

            lr = get_lr(epoch * iters + step, epochs * iters, args.learning_rate)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            with autocast_ctx:
                with torch.no_grad():
                    teacher_out = teacher(input_ids)
                    teacher_logits = teacher_out.logits.detach()

                student_out  = student(input_ids, labels=labels)
                total_loss, hard_val, soft_val = distillation_loss(
                    student_out.logits, teacher_logits, labels,
                    temperature=args.temperature, alpha=args.alpha,
                )
                total_loss = total_loss / args.accumulation_steps

            scaler.scale(total_loss).backward()

            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(student.parameters(), args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            cur = total_loss.item() * args.accumulation_steps
            loss_history.append(cur)
            hard_history.append(hard_val)
            soft_history.append(soft_val)
            step_history.append(global_step)
            pbar.set_postfix({
                'total': f'{cur:.4f}',
                'hard':  f'{hard_val:.4f}',
                'soft':  f'{soft_val:.4f}',
                'lr':    f'{lr:.2e}',
            })

            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
                ax1.plot(step_history, loss_history, 'b-', alpha=0.5, linewidth=0.8)
                if len(loss_history) > 20:
                    w  = 20
                    sm = [sum(loss_history[max(0,i-w):i+1]) / min(i+1, w)
                          for i in range(len(loss_history))]
                    ax1.plot(step_history, sm, 'r-', linewidth=2, label='Smoothed')
                ax1.set_title(f'Total Loss (step {global_step})')
                ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
                ax1.legend(); ax1.grid(True, alpha=0.3)

                ax2.plot(step_history, hard_history, 'g-', alpha=0.6,
                         linewidth=0.8, label=f'Hard (α={args.alpha})')
                ax2.plot(step_history, soft_history, 'm-', alpha=0.6,
                         linewidth=0.8, label=f'Soft (1-α={1-args.alpha})')
                ax2.set_title('Hard vs Soft Loss Components')
                ax2.set_xlabel('Step'); ax2.legend(); ax2.grid(True, alpha=0.3)
                plt.tight_layout(); plt.show()

distill_train(args.epochs)


In [ ]:
student.eval()
ckp = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}.pth'
torch.save({k: v.half().cpu() for k, v in student.state_dict().items()}, ckp)
print(f"✅ Distilled student saved: {ckp}")


## ⚖️ Teacher vs Student Comparison

Since the student was *never shown ground-truth labels for many steps*,  
it must reconstruct correct behaviour purely from the teacher's soft guidance.

In [ ]:
def generate(m, tok, prompt, device='cuda', max_new_tokens=120):
    m.eval()
    conv  = [{"role": "user", "content": prompt}]
    text  = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    inp   = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(inp["input_ids"], attention_mask=inp["attention_mask"],
                         max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, pad_token_id=tok.pad_token_id,
                         eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

test_prompts = [
    "用简单的语言解释什么是神经网络",
    "深度学习和机器学习有什么区别？",
    "推荐学习人工智能的路径",
]

for p in test_prompts:
    print(f"\n{'='*60}")
    print(f"❓ {p}")
    print(f"🟡 Teacher (SFT) : {generate(teacher, tokenizer, p, args.device)[:220]}")
    print(f"🟢 Student (Dist): {generate(student, tokenizer, p, args.device)[:220]}")


In [ ]:
import gc, torch

del teacher
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"Freed teacher GPU memory. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 📝 Student Exercise

1. **Vary alpha**: Train three runs with `alpha=0.1`, `0.5`, `0.9`.  
   Which produces better qualitative results? When is a higher soft-loss weight beneficial?

2. **Vary temperature**: Try `T=1.0` (standard CE from teacher) vs `T=4.0`.  
   Plot the resulting probability distributions on a toy 5-token vocabulary (like the visualisation above).

3. **Layer-wise distillation**: Instead of only distilling the final logits, distil  
   intermediate **hidden states** as well. Modify the training loop to add an L2 loss between  
   matching transformer layers. Does this improve results?

## 💡 Discussion Questions

- In the original Hinton et al. (2015) distillation paper, temperature was the key innovation.  
  Why does $T^2$ appear as a scaling factor in the soft-loss term?
- Our teacher and student have the **same size**. When is same-size distillation still useful?
- How is **data augmentation distillation** different from the approach we used?
- Compare distillation with **LoRA from Module 6** as strategies for producing a capable small model.
